In [ ]:
# library imports
import cartopy
import cmocean
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import os
import warnings
import glob
from netCDF4 import Dataset
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
# Sentinel-6 altimetry data comes in two modes: Low Resolution (LR, Pulse-Limited o LRM) and High Resolution (HR, Delay-Doppler). 
# We will inspect how these modes affect the Level-1B radar waveforms.

# Local file paths
LR_path = "S6A_P4_1B_LR______20210116T040229_20210116T045841_20250531T120010_3372_007_001_127_EUM__REP_NT_G01.SEN6.measurement.nc"
HR_path = "S6A_P4_1B_HR______20210116T040229_20210116T045841_20250531T142048_3372_007_001_127_EUM__REP_NT_G01.SEN6.measurement.nc"

In [ ]:
# Run this cell to inspect the hierarchical architecture of the Sentinel-6 NetCDF4 file.
#
# 1. Look for 'group: data_20': This contains the high-rate (~20 Hz) data.
# 2. Look for 'group: ku' inside it: This holds the Ku-band variables we need.

!ncdump -h S6A_P4_1B_LR______20210116T040229_20210116T045841_20250531T120010_3372_007_001_127_EUM__REP_NT_G01.SEN6.measurement.nc

In [ ]:
# Do the same for the other file (HR)

??

In [ ]:
# EXTRACTING WAVEFORMS FROM LR AND HR PRODUCTS

# We extract the 'power_waveform' array from the high-rate 20 Hz Ku-band 
# subgroups ('data_20/ku') for both operational modes.

# --- Extract from Low Resolution Mode (LRM) File ---
try:
    with Dataset(LR_path, "r") as ds:
        # Navigate the hierarchical group structure
        grp = ds.groups["data_20"]
        sub_grp = grp.groups["ku"]

        power_wf_lr = sub_grp.variables["power_waveform"][:]
            
except Exception as e:
    print(f"Error processing file {LR_path}: {e}")

# --- Extract from High Resolution Mode (SAR) File ---
try:
    with Dataset(??, "r") as ??:
        # Navigate the identical hierarchical structure in the HR file
        ??
        ??

        power_wf_hr = ??
            
except Exception as e:
    print(f"Error processing file {??}: {e}")

In [ ]:
# Plot the results

# Select a representative along-track waveform index to inspect
wf_index = 3000

fig = plt.figure(figsize=(10, 4), dpi=300)
ax = plt.subplot(1, 1, 1)

ax.plot(power_wf_lr[wf_index, :], color='blue', label='LR mode', linewidth=1.5)
ax.plot(power_wf_hr[wf_index, :], color='red', label='SAR (HR) mode', linewidth=1.5)

# Labeling and styling
ax.set_xlabel('Gate Index (Automatic Native Indexing)', fontsize=11)
ax.set_ylabel('Amplitude [counts]', fontsize=11)
ax.set_title(f"Waveform Comparison at Profile Index {wf_index}", fontsize=12, fontweight='bold')

ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Before analyzing how different surfaces affect the radar echo, we need to 
# understand the internal mapping of the 'surface_classification_flag'.

# RUN THIS COMMAND

!ncdump -h S6A_P4_1B_LR______20210116T040229_20210116T045841_20250531T120010_3372_007_001_127_EUM__REP_NT_G01.SEN6.measurement.nc | grep -A 8 "surface_classification_flag"

In [ ]:
# SURFACE CLASSIFICATION WAVEFORM AVERAGING 
# Different types of surface produce different waveforms. Using the surface classification information 
# provided in the Sentinel-6 level-1B altimetry products, the cell below plots a selection of waveforms associated 
# with different surface classes.


# EXERCISE: Extract the surface classification flags and power waveforms 
# from the Ku-band subgroup to plot average waveforms for different surface types.

try:
    with Dataset(LR_path, "r") as ds_lr_file:
        # Navigate to the 20Hz Ku-band subgroup
        ??
        ??
        
        surf = ?? #surface classification flag extraction
        wf   = ?? #power waveform extraction

    # Plotting setup
    fig = plt.figure(figsize=(12, 6), dpi=300)
    ax = plt.subplot(1, 1, 1)
    cmap = plt.cm.magma

    p1, = ax.plot(np.nanmean(wf[np.where(surf == 0)[0][0:100], :], axis=0), c=cmap(0.0),  label='Ocean')
    p2, = ax.plot(np.nanmean(wf[np.where(surf == 1)[0][0:100], :], axis=0), c=cmap(0.16), label='Land', linestyle='--')
    p3, = ax.plot(np.nanmean(wf[np.where(surf == 2)[0][0:100], :], axis=0), c=cmap(0.33), label='Continental water')
    p4, = ax.plot(np.nanmean(wf[np.where(surf == 3)[0][0:100], :], axis=0), c=cmap(0.50), label='Aquatic vegetation', linestyle='--')
    p5, = ax.plot(np.nanmean(wf[np.where(surf == 4)[0][0:100], :], axis=0), c=cmap(0.66), label='Continental ice/snow')
    p6, = ax.plot(np.nanmean(wf[np.where(surf == 5)[0][0:100], :], axis=0), c=cmap(0.84), label='Floating ice', linestyle='--')
    p7, = ax.plot(np.nanmean(wf[np.where(surf == 6)[0][0:100], :], axis=0), c=cmap(0.99), label='Salted basin')

    ax.set_xlabel('Gate Number (Time)', fontsize=14)
    ax.set_ylabel('Amplitude [counts]', fontsize=14)
    ax.set_title('Average Waveforms by Surface Classification Type', fontsize=16)
    
    ax.legend(loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.show()

except Exception as e:
    print(f"Error during surface classification plotting: {e}")

In [ ]:
# 3D WAVEFORM PROFILE 

#  We can also look at the variability in the waveforms along the track. The cell below takes a waveform every 
#  so often along the whole track and plot it.

# Configuration Switch: 0 = Low Resolution (LRM), 1 = High Resolution / SAR (HR)
high_res = 1
stride = 100  # Grouping factor: averages 100 along-track profiles into a single slice
cmap = plt.cm.RdYlBu

# Select file path based on the chosen mode
current_path = HR_path if high_res == 1 else LR_path

try:
    with Dataset(current_path, "r") as ds:
        # Navigate to the high-rate 20Hz Ku-band subgroup
        ku_20 = ds.groups["data_20"].groups["ku"]

        power_wf = ku_20.variables["power_waveform"][:]
        all_lats = ku_20.variables["latitude"][:]
        all_lons = ku_20.variables["longitude"][:]
        
        # GEOGRAPHIC CORRECTION: Convert longitudes from [0, 360) to [-180, 180) 
        # to prevent Cartopy from misinterpreting the boundaries of the track.
        all_lons[all_lons > 180] = all_lons[all_lons > 180] - 360

    # Determine data dimensions and dynamically generate the gate index axis (X-axis)
    num_records, num_gates = power_wf.shape
    samples = np.arange(num_gates) # 256 for LR, 512 for HR due to processing oversampling
    
    # Pre-allocate arrays to store downsampled along-track profiles
    counter = int(num_records / stride)
    waveform_array = np.zeros((counter, len(samples)))
    lon_array = []
    lat_array = []

    # Figure initialization
    fig = plt.figure(figsize=(12, 10), dpi=300)
    
    # --- SUBPLOT 1: Main 3D Plot ---
    ax1 = fig.add_subplot(1, 1, 1, projection='3d')

    for ii in range(counter):
        # Compute indices for the current along-track window
        idx_start = ii * stride
        idx_end   = (ii + 1) * stride
        
        # Calculate the spatial mean waveform and mean coordinates for this block
        waveform_array[ii, :] = np.nanmean(power_wf[idx_start:idx_end, :], axis=0)
        lons = np.nanmean(all_lons[idx_start:idx_end])
        lats = np.nanmean(all_lats[idx_start:idx_end])
        
        lon_array.append(lons)
        lat_array.append(lats)
        
        ax1.plot3D(samples, np.ones(len(samples)) * lats, waveform_array[ii, :], 
                   color=cmap(ii / counter), linewidth=0.5, zorder=counter - ii)

    # 3D Axis styling
    ax1.view_init(elev=20, azim=245)
    ax1.set_xlabel('Gate Number', fontweight='bold', labelpad=10)
    ax1.set_ylabel('Latitude', fontweight='bold', labelpad=10)
    ax1.set_zlabel('Amplitude [counts]', fontweight='bold', labelpad=10)
    ax1.set_title(f"3D Wavefall Transition - Mode: {'HR (SAR)' if high_res==1 else 'LR'}", 
                 fontweight='bold', fontsize=14, pad=20)

    # --- SUBPLOT 2: Inset 2D Spatial Track Map ---
    ax2 = fig.add_axes([0.68, 0.62, 0.22, 0.22], projection=ccrs.PlateCarree())

    ax2.scatter(lon_array, lat_array, 3, range(len(lon_array)), cmap=cmap, transform=ccrs.PlateCarree(), zorder=3)
    ax2.add_feature(cfeature.LAND, zorder=1, edgecolor='k', facecolor='#5D6D7E')
    ax2.add_feature(cfeature.OCEAN, zorder=0, facecolor='#AEB6BF') 

    buffer = 2.0
    ax2.set_extent([min(lon_array)-buffer, max(lon_array)+buffer, min(lat_array)-buffer, max(lat_array)+buffer], crs=ccrs.PlateCarree())

    # Coordinate grid lines configuration
    g1 = ax2.gridlines(draw_labels=True, zorder=20, color='white', linestyle='--', linewidth=0.5)
    g1.top_labels = False
    g1.right_labels = False
    g1.xlabel_style = {'color': 'black', 'size': 7}
    g1.ylabel_style = {'color': 'black', 'size': 7}

    plt.show()

except Exception as e:
    print(f"Error executing 3D waterfall processing: {e}")